# Benchmark: where Nemotron struggles, and how other models compare

The same agent tasks on each model: 5 decision scenarios, a prompt-injection attack, 4 analyst questions, and a raw honesty probe with no guardrails. Every guardrail refusal is counted by kind, so we see what each model *tried* to do wrong.

Cost: roughly 5-10 cents per model. Ultra is slow (several minutes); run it last.

In [ ]:
# 1. Setup
import os, sys
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (no credits)
from fieldvision_agents.benchmark import test_benchmark as t
for name, fn in vars(t).items():
    if name.startswith('test_'):
        fn(); print('PASS', name)

In [ ]:
# 3. Which models can we compare? (tiny tool-calling ping each, ~free)
from fieldvision_agents.benchmark.benchmark import NEMOTRON, discover_models, supports_tools
others = discover_models()
candidates = NEMOTRON + others
for m in candidates:
    print(supports_tools(m), m)

In [ ]:
# 4. Choose models: tool-calling ones only. Ultra last because it is slow.
usable = [m for m in candidates if supports_tools(m)[0]]
fast_first = [m for m in usable if 'ultra' not in m] + [m for m in usable if 'ultra' in m]
fast_first

In [ ]:
# 5. Run the benchmark (one model at a time; re-run the cell after edits, results are cached)
from IPython.display import Markdown, display
from fieldvision_agents.benchmark.benchmark import compare
bench = compare([m for m in fast_first if 'ultra' not in m])
display(Markdown(bench['markdown']))
print('saved to', bench['path'])

In [ ]:
# 6. Optional: add Ultra (several minutes)
ultra = [m for m in fast_first if 'ultra' in m]
if ultra:
    bench_u = compare(ultra)
    bench['rows'] += bench_u['rows']
    from fieldvision_agents.benchmark.benchmark import markdown_table
    display(Markdown(markdown_table(bench['rows'])))

In [ ]:
# 7. Limitations table for the slide (what we saw while building + what the benchmark measured)
from fieldvision_agents.benchmark.benchmark import limitations_markdown
lim = limitations_markdown(bench['rows'])
Path('benchmark_results').mkdir(exist_ok=True)
Path('benchmark_results/limitations.md').write_text(lim + '\n')
display(Markdown(lim))

In [ ]:
# 8. Look closer: raw answers and errors per model
for r in bench['rows']:
    print('==', r['model'], '| errors:', r['errors'] or 'none')
    print('   raw:', (r.get('raw') or {}).get('answer', '')[:400], '\n')